In [1]:
import hail as hl
#import argparse
#import pandas
import os

from ukb_utils import hail_init
from ukb_utils import genotypes
from ko_utils import qc
from ko_utils import analysis

In [2]:
import hail as hl
from gnomad.utils.vep import process_consequences

In [3]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('/logs/hail/hail_export.log', 'GRCh38')

Running on Apache Spark version 3.1.2
SparkUI available at http://compe074.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.74-0c3a74d12093
LOGGING: writing to /logs/hail/hail_export.log


In [ ]:
# get non singletons (note: individuals are on 12788)
mt1 = qc.get_table('data/phased/ukb_wes_200k_phased_chr22.1of1.vcf.gz', 'vcf')
#mt1 = qc.get_table('data/tmp/ukb_wes_200k_phased_tmp_chr22.1of1.vcf.gz', 'vcf')
#mt1 = qc.recalc_info(mt1)
mt1 = qc.translate_sample_ids(mt1, from_app='12788', to_app='11867') # translate to lindgren app ID
#mt1 = qc.filter_to_european(mt1)
mt1 = qc.filter_min_missing(mt1, 0.05)
#mt1 = qc.filter_max_maf(mt1, float(0.02))
mt1 = analysis.annotate_vep(mt1, vep_path = 'data/vep/full/ukb_wes_200k_full_vep_chr22.vcf')
#mt1 = analysis.filter_vep(mt1, 'consequence_category',['damaging_missense','ptv'])
#mt1 = mt1.filter_rows(hl.agg.any(mt1.GT.is_non_ref()))

In [ ]:
def annotate_phased_entries(mt):
    r'''Annotates alleles that have the alternate allele on either first or second strand.'''
    mt = mt.annotate_entries(a0_alt = mt.GT ==  hl.parse_call('1|0'))
    mt = mt.annotate_entries(a1_alt = mt.GT ==  hl.parse_call('0|1'))
    mt = mt.annotate_entries(a_homo = mt.GT ==  hl.parse_call('1|1'))
    return mt


In [ ]:
mt1 = annotate_phased_entries(mt1)

In [ ]:
def annotate_compound_hetz(mt):
    
    mt = mt.filter_entries(~mt.GT.is_hom_var())
    ht0 = (mt.group_rows_by(mt.vep.Gene)
           .aggregate(s0 = hl.agg.any(mt.a0_alt))).entries()
    ht1 = (mt.group_rows_by(mt.vep.Gene)
           .aggregate(s1 = hl.agg.any(mt.a1_alt))).entries()
    ht = ht0.annotate(s1 = ht1[ht0.Gene, ht0.s].s1)
    ht = ht.filter((ht.s0 == True) & (ht.s1 == True))
    
    return None
    


In [ ]:
mt = mt1
compound_heterozygous = mt.a0_alt & mt.a0_alt
homozygous = mt.a_homo
mt = mt.filter_entries(~mt.GT.is_hom_var())

ht0 = (mt.group_rows_by(mt.vep.Gene)
        .aggregate(s0 = hl.agg.any(mt.a0_alt))).entries()
ht1 = (mt.group_rows_by(mt.vep.Gene)
        .aggregate(s1 = hl.agg.any(mt.a1_alt))).entries()

In [ ]:
ht = ht0.annotate(s1 = ht1[ht0.Gene, ht0.s].s1)
#ht.describe()
ht = ht.filter((ht.s0 == True) & (ht.s1 == True))
ht.show()

In [ ]:
mt2 = qc.get_table('data/unphased/unfiltered/ukb_wes_200k_filtered_chr22.mt','mt')
#mt2 = qc.get_table('data/unphased/tmp/ukb_wes_200k_filtered_2samples_chr22.vcf.bgz', 'vcf')
mt2 = qc.filter_to_european(mt2)
mt2 = qc.filter_max_mac(mt2, 1) # get singletons
mt2 = qc.filter_min_missing(mt2, 0.05)
mt2 = analysis.annotate_vep(mt2, vep_path = 'data/vep/full/ukb_wes_200k_full_vep_chr22.vcf')
mt2 = analysis.filter_vep(mt2, 'consequence_category',['damaging_missense','ptv'])


In [ ]:
out = analysis.construct_summary_mt(mt1)

In [ ]:
out.aggregate_entries(hl.agg.counter(out.ko))

In [ ]:
# Count burden per gene per individual
mt1_cat = analysis.gene_burden_category_annotations_per_sample(mt1)
mt2_cat = analysis.gene_burden_category_annotations_per_sample(mt2)

# combine singleton table and full table
res = mt1_cat.annotate_entries(singletons = mt2_cat[(mt1_cat.Gene, mt1_cat.consequence_category), mt1_cat.s].n)
res = res.annotate_entries(singletons = hl.if_else(hl.is_missing(res.singletons),0,res.singletons))
res = res.annotate_entries(total = res.n + res.singletons)
res = res.entries()
res = res.filter(res.total > 0)

In [ ]:
n = res.singletons.collect()

In [ ]:
n

In [ ]:
# why are there no singletons left in chr 22 after filtering?

# How many singletons / individuals at missingness 5%
mmt2 = mt2.filter_rows(hl.agg.any(mt2.GT.is_non_ref()))
#mmt1 = mt1.filter_rows(hl.agg.any(mt1.GT.is_non_ref()))

# 

In [ ]:
mmt2.count()

In [ ]:
# Count burden per gene per individual
mt1_cat = analysis.gene_burden_category_annotations_per_sample(mt1)
mt2_cat = analysis.gene_burden_category_annotations_per_sample(mt2)

# combine singleton table and full table
res = mt1_cat.annotate_entries(singletons = mt2_cat[(mt1_cat.Gene, mt1_cat.consequence_category), mt1_cat.s].n)
res = res.annotate_entries(singletons = hl.if_else(hl.is_missing(res.singletons),0,res.singletons))
res = res.annotate_entries(total = res.n + hl.if_else(hl.is_missing(res.singletons),0,res.singletons))
res = res.entries()
res = res.filter(res.singletons > 0)

In [ ]:
mt = analysis.get_prob_ko_matrix(mt1, mt2, field_drop = None)

In [ ]:
mt = qc.get_table('data/mt/ukb_wes_vep_200_chr22.mt','mt')
mt = analysis.annotate_vep(mt, vep_path = 'data/vep/full/ukb_wes_200k_full_vep_chr22.vcf')

In [ ]:
mt.vep.decribe()

In [4]:
# annotate with VEP
#hail_init.hail_bmrc_init('/logs/hail/hail_vep_export.log', 'GRCh38')
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
dataset = qc.get_table('data/unphased/unfiltered/ukb_wes_200k_filtered_chr2.mt', 'mt')
    
# clean up snpID and rsID
dataset = qc.annotate_snpid(dataset)
dataset = qc.annotate_rsid(dataset)
dataset = qc.default_to_snpid_when_missing_rsid(dataset)

# Translate to lindgren IDs
#if True:
#    dataset = qc.translate_sample_ids(dataset, 12788, 11867)    
    
# recalc info
dataset = qc.recalc_info(dataset)

# get VEP
result = hl.vep(dataset, "utils/configs/vep_env.json") 
result = process_consequences(result)
#qc.export_table(result, out_prefix = 'testtabledeleteme', out_type = 'mt')


2021-09-21 09:23:50 Hail: WARN: expected input file 'file:/well/lindgren/flassen/ressources/dbsnp/GRCh38/GCF_000001405.39.gz' to end in .vcf[.bgz, .gz]


In [ ]:
def filter_european(mt, only_annotate = False):
    r'''Get white british (app 11867) /well/lindgren/UKBIOBANK/DATA/QC/ukb_sqc_v2.txt
    and genetically european from /well/lindgren/UKBIOBANK/laura/k_means_clustering_pcs/ukbb_genetically_european_k4_4PCs_self_rep_Nov2020.txt''' 
    ht = hl.import_table('/well/lindgren/UKBIOBANK/laura/k_means_clustering_pcs/ukbb_genetically_european_k4_4PCs_self_rep_Nov2020.txt',
        types={'eid': hl.tstr, 'genetically_european': hl.tint32}).key_by('eid')
    mt = mt.annotate_cols(eur = ht[mt.s].genetically_european)
    undefined_eur = mt.aggregate_cols(hl.agg.sum(hl.is_missing(mt.eur)))
    pre_filter_count = mt.count()
    if undefined_eur == pre_filter_count[1]:
        raise ValueError('[get_genetically_european]: IDs for genetically europeans does not match keys in MatrixTable!')
    if undefined_eur > 0:
        print(f'[get_genetically_european]: Not all samples IDs mapped perfectly ({undefined_eur}/{pre_filter_count[1]} IDs are undefined)')
    if only_annotate == False:
        mt = mt.filter_cols(mt.eur == 1)
        post_filter_count = mt.count()
        print(f'[get_genetically_european]:{post_filter_count[1]}/{pre_filter_count[1]} IDs were included as genetically european.')
    return mt

In [8]:
result.vep.worst_consequence_by

AttributeError: StructExpression instance has no field, method, or property 'worst_consequence_by'
    Did you mean:
        Data fields: 'worst_consequence_term', 'transcript_consequences', 'most_severe_consequence', 'worst_csq_by_gene', 'motif_feature_consequences'

In [15]:
result.vep.worst_csq_by_gene_canonical.describe()

--------------------------------------------------------
Type:
        array<struct {
        allele_num: int32, 
        amino_acids: str, 
        biotype: str, 
        canonical: int32, 
        ccds: str, 
        cdna_start: int32, 
        cdna_end: int32, 
        cds_end: int32, 
        cds_start: int32, 
        codons: str, 
        consequence_terms: array<str>, 
        distance: int32, 
        domains: array<struct {
            db: str, 
            name: str
        }>, 
        exon: str, 
        gene_id: str, 
        gene_pheno: int32, 
        gene_symbol: str, 
        gene_symbol_source: str, 
        hgnc_id: str, 
        hgvsc: str, 
        hgvsp: str, 
        hgvs_offset: int32, 
        impact: str, 
        intron: str, 
        lof: str, 
        lof_flags: str, 
        lof_filter: str, 
        lof_info: str, 
        minimised: int32, 
        polyphen_prediction: str, 
        polyphen_score: float64, 
        protein_end: int32, 
        protein_s

In [ ]:
#mt.describe()

In [ ]:

#mt.annotate_rows(dbNSFP = )

In [ ]:
#mt.vep.worst_csq_by_gene.gene_symbol.show()

In [15]:
def annotate_dbnsfp(mt, vep_path):
    r'''Annotate matrix table with dbNSFP consequence from external VEP file.'''
    print(f'Annotating with VEP file: {vep_path}')
    
    # Open file containing VEP fields
    with open('data/vep/vep_fields.txt', 'r') as file:
        fields = file.read().strip().split(',')
    ht = hl.import_vcf(vep_path).rename({'info':'vep'}) 
    
    # Add VEP fields by iteration
    for i in range(len(fields)):
        ht = ht.annotate_rows(
            vep=ht.vep.annotate(
                col=ht.vep.CSQ.map(lambda x: (x.split('\\|')[i]))[0]
                ).rename({'col':f'{fields[i]}'})
        )
    
    # Extract various categories annotations and change type
    ht = ht.annotate_rows(vep = ht.vep.annotate(sift_pred = ht.vep.SIFT_pred.split('&')[0]))
    ht = ht.annotate_rows(vep = ht.vep.annotate(polyphen2_hdiv_pred = ht.vep.Polyphen2_HDIV_pred.split('&')[0]))
    ht = ht.annotate_rows(vep = ht.vep.annotate(polyphen2_hvar_pred = ht.vep.Polyphen2_HVAR_pred.split('&')[0]))
    ht = ht.annotate_rows(vep = ht.vep.annotate(cadd_phred_score = hl.parse_float(ht.vep.CADD_phred)))
    ht = ht.annotate_rows(vep = ht.vep.annotate(revel_score = hl.parse_float(ht.vep.REVEL_score)))
    
    # annotate main table
    mt = mt.annotate_rows(dbnsfp = hl.struct())
    mt = mt.annotate_rows(dbnsfp = mt.dbnsfp.annotate(revel_score = ht.index_rows(mt.locus, mt.alleles).vep.revel_score))
    mt = mt.annotate_rows(dbnsfp = mt.dbnsfp.annotate(cadd_phred_score = ht.index_rows(mt.locus, mt.alleles).vep.cadd_phred_score))
    mt = mt.annotate_rows(dbnsfp = mt.dbnsfp.annotate(polyphen2_hdiv_pred = ht.index_rows(mt.locus, mt.alleles).vep.polyphen2_hdiv_pred))
    mt = mt.annotate_rows(dbnsfp = mt.dbnsfp.annotate(polyphen2_hvar_pred = ht.index_rows(mt.locus, mt.alleles).vep.polyphen2_hvar_pred))

    return(mt)

ht = annotate_dbnsfp(mt, vep_path = 'data/vep/full/ukb_wes_200k_full_vep_chr22.vcf')
ht.vep.consequence_category

Annotating with VEP file: data/vep/full/ukb_wes_200k_full_vep_chr22.vcf


AttributeError: StructExpression instance has no field, method, or property 'consequence_category'
    Did you mean:
        Data field: 'worst_consequence_term'

In [16]:
mt = qc.get_table('data/mt/ukb_wes_vep_200_chr22.mt', 'mt')
mt = annotate_dbnsfp(mt, vep_path = 'data/vep/full/ukb_wes_200k_full_vep_chr22.vcf')
mt = variant_csqs_builder(mt)

Annotating with VEP file: data/vep/full/ukb_wes_200k_full_vep_chr22.vcf


In [ ]:
PLOF_CSQS = ["transcript_ablation", "splice_acceptor_variant",
             "splice_donor_variant", "stop_gained", "frameshift_variant"]

MISSENSE_CSQS = ["stop_lost", "start_lost", "transcript_amplification",
                 "inframe_insertion", "inframe_deletion", "missense_variant"]

SYNONYMOUS_CSQS = ["stop_retained_variant", "synonymous_variant"]

OTHER_CSQS = ["mature_miRNA_variant", "5_prime_UTR_variant",
              "3_prime_UTR_variant", "non_coding_transcript_exon_variant", "intron_variant",
              "NMD_transcript_variant", "non_coding_transcript_variant", "upstream_gene_variant",
              "downstream_gene_variant", "TFBS_ablation", "TFBS_amplification", "TF_binding_site_variant",
              "regulatory_region_ablation", "regulatory_region_amplification", "feature_elongation",
              "regulatory_region_variant", "feature_truncation", "intergenic_variant"]

def variant_csqs_category_builder(mt):
    r'''Create categories for downstream analysis'''
    return mt.annotate_rows(vep = mt.vep.annotate(consequence_category = 
        hl.case().when(PLOF_CSQS.contains(mt.vep.most_severe_consequence), "ptv")
             .when(MISSENSE_CSQS.contains(mt.vep.most_severe_consequence) & 
                   (~hl.is_defined(mt.dbnsfp.cadd_phred_score) | 
                    ~hl.is_defined(mt.dbnsfp.revel_score)), "other_missense")                                   
             .when(MISSENSE_CSQS.contains(mt.vep.most_severe_consequence) & 
                   (mt.dbnsfp.cadd_phred_score >= 20) & 
                   (mt.dbnsfp.revel_score >= 0.6), "damaging_missense") 
             .when(MISSENSE_CSQS.contains(mt.vep.most_severe_consequence), "other_missense")
             .when(SYNONYMOUS_CSQS.contains(mt.vep.most_severe_consequence), "synonymous")
             .when(OTHER_CSQS.contains(mt.vep.most_severe_consequence), "non_coding")
             .default("NA")))